# Results for model: microsoft/phi-3-mini-4k-instruct

In [1]:
import xgboost as xgb
import numpy as np
import pandas as pd
from polars import read_parquet

# Constants for file paths and parameters
DATA_PATH = './jane_street_data/train.parquet'
DAYS_BACK_PERIOD = 14
FEATURE_COLUMN = 'feature_00'
TARGET_COLUMN = 'responder_6'
TOP_QUANTILE_PARTITIONS = 0.15

# Load the data using polars
df = read_parquet(DATA_PATH)

# Convert to pandas DataFrame if necessary for xgboost (if needed)
df_pandas = df.to_pandas()

# Feature Engineering: Global TOP_QUANTILE (top 15%) of feature_00
def calc_top_quantile(df, feature, target, period, partitions=TOP_QUANTILE_PARTITIONS):
    df['date_order'] = df['date_id']
    df = df.sort({'date_order': ascending=True})
  
    df['rolling_quantile'] = df.with_column(
        xgb.quantile(
            df[target],
            q=partitions,
            group_cols=['feature', 'date_order'],
            order_by=['date_order']
        )
    ).groupby([feature, 'date_order']).transform('first')
    return df

df_engineered = calc_top_quantile(df_pandas, FEATURE_COLUMN, TARGET_COLUMN, DAYS_BACK_PERIOD)

input_features = [F"{FEATURE_COLUMN}[{i}]" for i in range(DAYS_BACK_PERIOD)] + [TARGET_COLUMN, 'date_id']
input_df = df_engineered[input_features]

train_data = xgb.DMatrix(input_df.values, label=df_engineered[TARGET_COLUMN].values)

# Define the XGBoost regressor parameters
params = {
    'objective': 'reg:squarederror',
    'eval_metric': 'rmse',
    'eta': 0.025,
    'max_depth': 8,
    'min_child_weight': 3
}

# Train the XGBoost Regressor
bst = xgb.train(params, train_data, num_boost_round=100)

# Save the model
bst.save_model('jane_street_market_xgb.model')

SyntaxError: invalid syntax (1784465999.py, line 22)